# R5.1 — Which part of the detector carries the signal?

Reviewer 5, comment 1: compare prediction disagreement of (i) ordinary separately trained models,
(ii) contrastive Siamese models, and (iii) an explicit embedding-level measure.

Both variants share the published noisy labels and outer folds (rebuilt from the committed
prediction CSVs), the same seeded inner split, the same 200k training pairs per member, the same
initial weights, schedule and early stopping. They differ only in the objective:

| variant | objective |
|---|---|
| `siamese` | ours: CE on both branches + the contrastive loss `same·d² + (1-same)·[m-d]₊²` (`snd.training.contrastive`, the corrected form used for the CIFAR-10 runs) |
| `ce` | CE only (contrastive weight 0): an ordinary classifier ensemble on identical data |
| `ce_linear` | optional: backbone + plain linear head, CE only |

Each member saves, for the held-out outer fold, its softmax outputs and embeddings, and the
embeddings of its own training subset. The analysis then scores every held-out sample by
disagreement (the published detector), mean confidence in the observed label, and three
embedding-level scores computed inside each member (centroid margin, k-NN label disagreement,
across-member instability of the distance to the observed-label centroid; embedding spaces of
separately trained members are not aligned, so raw embedding variance is not meaningful).
It also reports the per-member misclassification rate on noisy vs clean samples (R4.3) and
paired-bootstrap CIs for every AUC difference.

**Budget (rough, refine from the first member's log):** CIFAR-10 ResNet-50 ~1.2–1.6 h per member
on a T4 with AMP, so one outer fold × 2 variants × 10 members ≈ 28 GPU-h ≈ 14 h on T4×2
(two commits). Fashion-MNIST ResNet-34 ≈ 0.3 h per member, so one outer fold ≈ 3–4 h on T4×2.

**Kaggle settings:** Accelerator *GPU T4 x2* (P100 also works, one worker), Internet *On*
(phone-verified account), Persistence *Files only*. Run with **Save Version -> Save & Run All
(Commit)** so it runs in the background for up to 12 h.

**Resuming:** jobs are saved as they finish. If the session ends with jobs left, open the
notebook, *Add Input -> Your Work -> this notebook* (the previous version's output), and
commit again: finished jobs are restored and skipped. Repeat until the launch cell prints
that nothing is left; the analysis cell then reports the final tables.

**Outputs** (in the version's Output tab): `r51_out/` raw per-job results, `results/` CSV
tables, `logs/` per-GPU training logs.

## 1. Configuration

In [ ]:
import json, os
CONFIG = dict(
    dataset='cifar10', noise=20,      # ('cifar10', 20|30|40) or ('fashionmnist', 20|30|40)
    preds_ref='main',                 # run whose predictions are the reference row: 'main' = the
                                      # CIFAR-10 re-run, '513d833' = the round-3 paper run (same
                                      # noise draw and folds, so training here is unaffected)
    code_ref='main',                  # branch providing the snd package
    outer_folds=[1],                  # outer fold(s) to score; fold 1 is unaffected by the
                                      # inner-model reuse in the CIFAR-10 re-run (folds 4, 5, 8)
    variants=['siamese', 'ce'],       # add 'ce_linear' if budget allows
    members=10, seed=0, amp=True,
    session_hours=11.5, member_hours=1.6,        # don't start a member that would cross 11.5 h
    save_checkpoints=True, loader_workers=2,
)
CONFIG.update(json.loads(os.environ.get('SND_CONFIG_OVERRIDES', '{}')))   # local smoke tests only
OUT_NAME, WORKER, PIP = 'r51_out', 'r51_worker.py', ['timm']
N_JOBS = len(CONFIG['outer_folds']) * CONFIG['members'] * len(CONFIG['variants'])

## 2. Helper modules (written to the working directory)

In [ ]:
%%writefile snd_kaggle.py
"""Shared plumbing for the round-4 revision experiments (Kaggle or any CUDA box).

What lives here:
  * repo setup: clone the public SiameseNoiseDetection repo, import `snd` from it;
  * the exact noisy labels and outer folds of a published run, rebuilt from its
    committed prediction CSVs (the IDN generator re-randomises, so the CSVs are the
    only faithful record of which labels were corrupted);
  * the per-dataset protocol as actually run (main.ipynb cells 16/25/35, 59/71/80,
    106-124), which differs in places from snd/config.py and the CLI;
  * a crash-safe job queue that keeps every visible GPU busy and survives Kaggle's
    12-hour session limit by resuming from a previous version's output.
"""
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

REPO_URL = 'https://github.com/stormaref/SiameseNoiseDetection.git'
ON_KAGGLE = os.path.isdir('/kaggle/working')
WORK = '/kaggle/working' if ON_KAGGLE else os.path.abspath(os.environ.get('SND_WORK', 'work'))
# repo clone, extracted preds and datasets are re-creatable: keep them out of the saved output
SCRATCH = '/tmp/snd' if ON_KAGGLE else WORK
DATA_ROOT = os.path.join(SCRATCH, 'data')

CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)

# Siamese-ensemble protocol per published run (main.ipynb; values that differ
# from snd/config.py are noted). `aug` names an entry of `augmentation()` below.
PROTOCOL = {
    ('cifar10', 20): dict(preds='cifar10(20)', backbone='resnet50', pre_trained=True, emb=64,
                          wd=5e-4, patience=8, margin=2, aug='cifar_crop'),
    ('cifar10', 30): dict(preds='cifar10(30)', backbone='resnet50', pre_trained=True, emb=64,
                          wd=5e-4, patience=10, margin=2, aug='cifar_crop'),
    ('cifar10', 40): dict(preds='cifar10(40)', backbone='resnet50', pre_trained=True, emb=64,
                          wd=5e-4, patience=15, margin=2, aug='cifar_affine'),
    ('fashionmnist', 20): dict(preds='fmnist(20)', backbone='resnet34', pre_trained=False, emb=128,
                               wd=1e-3, patience=12, margin=2, aug='fmnist_plain'),
    ('fashionmnist', 30): dict(preds='fmnist(30)', backbone='resnet34', pre_trained=False, emb=128,
                               wd=1e-3, patience=12, margin=2, aug='fmnist_plain'),
    ('fashionmnist', 40): dict(preds='fmnist(40)', backbone='resnet34', pre_trained=False, emb=128,
                               wd=1e-3, patience=12, margin=2, aug='fmnist_norm'),
}
# Shared by every run above.
SIAMESE_COMMON = dict(lr=5e-5, batch_size=2048, train_pairs=200_000, val_pairs=20_000,
                      dropout=0.5, label_smoothing=0.1, max_epochs=1000, inner_folds=10)

# Downstream-classifier protocol (FinalEvaluator calls, main.ipynb cells 106-124).
DOWNSTREAM = {
    'cifar10': dict(lr=1e-3, wd=5.46e-5, batch_size=256, patience=20, warmup=5,
                    smoothing=0.1, val_ratio=0.1, max_epochs=200),
    'fashionmnist': dict(lr=2e-4, wd=1.12e-6, batch_size=256, patience=10, warmup=5,
                         smoothing=0.1, val_ratio=0.05, max_epochs=200),
}


# --------------------------------------------------------------------------- repo
def setup_repo(code_ref='main', repo_dir=None):
    """Clone (or reuse) the code repo, check out `code_ref`, make `snd` importable."""
    repo_dir = repo_dir or os.environ.get('SND_REPO') or os.path.join(SCRATCH, 'SiameseNoiseDetection')
    if not os.path.isdir(os.path.join(repo_dir, '.git')):
        os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, repo_dir], check=True)
    if not os.environ.get('SND_REPO'):      # never move the HEAD of a local working copy
        subprocess.run(['git', '-C', repo_dir, 'checkout', '--quiet', code_ref], check=True)
    # local testing: SND_SRC points at an extracted `src/` of code_ref when the working
    # copy in SND_REPO is on another branch
    src = os.environ.get('SND_SRC') or os.path.join(repo_dir, 'src')
    if src not in sys.path:
        sys.path.insert(0, src)
    return repo_dir


def _resolve_ref(repo_dir, ref):
    for candidate in (f'origin/{ref}', ref):
        ok = subprocess.run(['git', '-C', repo_dir, 'rev-parse', '--verify', '--quiet', candidate],
                            capture_output=True).returncode == 0
        if ok:
            return candidate
    raise ValueError(f'git ref {ref!r} not found in {repo_dir}')


def extract_preds(repo_dir, preds_ref, preds_name):
    """Materialise preds/<preds_name> as committed on branch `preds_ref`; return its dir."""
    out = os.path.join(SCRATCH, 'preds', preds_ref, preds_name)
    if not glob.glob(os.path.join(out, '*', 'fold*_analysis.csv')):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        ref = _resolve_ref(repo_dir, preds_ref)
        archive = subprocess.run(['git', '-C', repo_dir, 'archive', ref, f'preds/{preds_name}'],
                                 capture_output=True, check=True).stdout
        tmp = out + '.tmp'
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-x', '-C', tmp], input=archive, check=True)
        shutil.move(os.path.join(tmp, 'preds', preds_name), out)
        shutil.rmtree(tmp)
    return out


# --------------------------------------------------------------------------- labels
def load_label_table(preds_dir):
    """One row per training sample: noisy/true label, noise flag, outer fold, saved preds."""
    frames = []
    for path in glob.glob(os.path.join(preds_dir, '*', 'fold*_analysis.csv')):
        fold = int(re.search(r'fold(\d+)_analysis', path).group(1))
        df = pd.read_csv(path)
        df['outer_fold'] = fold
        frames.append(df)
    table = pd.concat(frames).set_index('index').sort_index()
    table['is_noisy'] = table['is_noisy'].astype(str).str.lower().eq('true')
    if not table.index.is_unique:
        raise ValueError(f'{preds_dir}: a sample appears in more than one outer fold')
    return table


def base_dataset(name, train=True, attempts=5):
    from torchvision.datasets import CIFAR10, FashionMNIST
    cls = {'cifar10': CIFAR10, 'fashionmnist': FashionMNIST}[name]
    for attempt in range(attempts):             # the mirrors fail transiently now and then
        try:
            return cls(root=DATA_ROOT, train=train, download=True)
        except RuntimeError as error:
            if attempt == attempts - 1:
                raise
            print(f'download failed ({error}); retrying in 30 s', flush=True)
            time.sleep(30)


def noisy_train_set(name, table):
    """The torchvision training set with its targets replaced by the published noisy labels."""
    ds = base_dataset(name, train=True)
    if len(table) != len(ds) or (table.index.to_numpy() != np.arange(len(ds))).any():
        raise ValueError('prediction CSVs do not cover the training set exactly once')
    true = np.asarray(ds.targets)
    if (table['real_label'].to_numpy() != true).any():
        raise ValueError('real_label in the CSVs does not match torchvision targets')
    ds.targets = [int(y) for y in table['noisy_label']]
    return ds


def smoke_subsample(table, cfg):
    """Local smoke tests only: a fixed random subset, re-indexed 0..n-1 (no-op on real runs)."""
    if not cfg.get('subsample'):
        return table, None
    keep = np.sort(np.random.default_rng(0).choice(len(table), cfg['subsample'], replace=False))
    return table.iloc[keep].reset_index(drop=True), keep


def augmentation(name):
    """Transforms exactly as in main.ipynb (scoring always uses `eval`)."""
    from torchvision import transforms as T
    cifar_norm = [T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)]
    gray = [T.Grayscale(num_output_channels=3), T.ToTensor()]
    table = {
        'cifar_crop': (T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(0.5),
                                  T.RandomRotation(15), *cifar_norm]),
                       T.Compose(cifar_norm)),
        'cifar_affine': (T.Compose([T.RandomRotation(15), T.RandomHorizontalFlip(0.5),
                                    T.RandomAffine(0, translate=(0.1, 0.1)),
                                    T.RandomResizedCrop(32, scale=(0.9, 1.0)), *cifar_norm]),
                         T.Compose(cifar_norm)),
        'fmnist_plain': (T.Compose(gray), T.Compose(gray)),
        'fmnist_norm': (T.Compose(gray + [T.Normalize((0.5,), (0.5,))]),
                        T.Compose(gray + [T.Normalize((0.5,), (0.5,))])),
        # downstream classifiers (FinalEvaluator cells)
        'down_cifar': (T.Compose([T.RandomRotation(15), T.RandomHorizontalFlip(0.5),
                                  T.RandomAffine(0, translate=(0.1, 0.1)),
                                  T.RandomResizedCrop(32, scale=(0.9, 1.0)), *cifar_norm]),
                       T.Compose(cifar_norm)),
        'down_fmnist': (T.Compose([T.Grayscale(num_output_channels=3), T.RandomHorizontalFlip(0.5),
                                   T.RandomRotation(10), T.RandomAffine(0, translate=(0.1, 0.1)),
                                   T.ToTensor(), T.Normalize((0.5,), (0.5,))]),
                        T.Compose(gray + [T.Normalize((0.5,), (0.5,))])),
    }
    return table[name]


def cleaned_labels(table, td, tr):
    """Apply the published DetectAndRelabel rule (NoiseCleaner.advanced_clean).

    Returns an int array of new labels with -1 for removed samples: flagged if
    mistakes >= td; relabelled to a class predicted by >= tr members, else removed.
    """
    labels = table['noisy_label'].to_numpy().copy()
    for pos, (mistakes, preds) in enumerate(zip(table['mistakes'], table['preds'])):
        if mistakes < td:
            continue
        votes = np.bincount(np.fromstring(preds, dtype=int, sep='|'), minlength=10)
        winners = np.flatnonzero(votes >= tr)
        labels[pos] = winners[0] if len(winners) else -1
    return labels


# --------------------------------------------------------------------------- jobs
class JobQueue:
    """File-backed queue: a job is done when its result file exists.

    Workers claim a job by atomically creating `<result>.claim`; claims left behind by
    a killed session are ignored once older than `stale_hours`.
    """

    def __init__(self, out_dir, stale_hours=13):
        self.out_dir = out_dir
        self.stale = stale_hours * 3600
        os.makedirs(out_dir, exist_ok=True)

    def done(self, job_id):
        return os.path.exists(os.path.join(self.out_dir, job_id + '.done'))

    def claim(self, job_id):
        path = os.path.join(self.out_dir, job_id + '.claim')
        if os.path.exists(path) and time.time() - os.path.getmtime(path) > self.stale:
            os.remove(path)
        try:
            fd = os.open(path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        except FileExistsError:
            return False
        os.write(fd, str(os.getpid()).encode())
        os.close(fd)
        return True

    def finish(self, job_id, summary):
        with open(os.path.join(self.out_dir, job_id + '.done'), 'w') as f:
            json.dump(summary, f, indent=1)
        claim = os.path.join(self.out_dir, job_id + '.claim')
        if os.path.exists(claim):
            os.remove(claim)

    def release(self, job_id):
        claim = os.path.join(self.out_dir, job_id + '.claim')
        if os.path.exists(claim):
            os.remove(claim)

    def summaries(self):
        rows = []
        for path in sorted(glob.glob(os.path.join(self.out_dir, '*.done'))):
            with open(path) as f:
                rows.append(json.load(f))
        return rows


def restore_previous_outputs(out_name):
    """Copy `<out_name>/` from any attached earlier notebook version into WORK.

    On Kaggle: Add Input -> Your Work -> this notebook (a previous version); its
    /kaggle/working appears under /kaggle/input/<slug>/.
    """
    target = os.path.join(WORK, out_name)
    os.makedirs(target, exist_ok=True)
    copied = 0
    for src in glob.glob(f'/kaggle/input/*/{out_name}') + glob.glob(f'/kaggle/input/*/*/{out_name}'):
        for path in glob.glob(os.path.join(src, '**', '*'), recursive=True):
            if os.path.isdir(path) or path.endswith('.claim'):
                continue
            dest = os.path.join(target, os.path.relpath(path, src))
            if not os.path.exists(dest):
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy2(path, dest)
                copied += 1
    print(f'restored {copied} files into {target}')
    return target


def launch_workers(script, config_path, n_gpus=None, log_dir=None, poll=60):
    """Run `python script config_path` once per GPU and stream a progress line per poll."""
    import torch
    n_gpus = n_gpus or max(torch.cuda.device_count(), 1)
    log_dir = log_dir or os.path.join(WORK, 'logs')
    os.makedirs(log_dir, exist_ok=True)
    procs = []
    for gpu in range(n_gpus):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), SND_WORKER=str(gpu))
        log = open(os.path.join(log_dir, f'worker{gpu}.log'), 'a')
        procs.append((subprocess.Popen([sys.executable, '-u', script, config_path], env=env,
                                       stdout=log, stderr=subprocess.STDOUT), log))
        time.sleep(20)                       # stagger dataset downloads / pair generation
    while any(p.poll() is None for p, _ in procs):
        time.sleep(poll)
        for gpu in range(n_gpus):
            tail = _last_line(os.path.join(log_dir, f'worker{gpu}.log'))
            print(f'[{time.strftime("%H:%M")}] gpu{gpu}: {tail}', flush=True)
    for p, log in procs:
        log.close()
    return [p.returncode for p, _ in procs]


def _last_line(path):
    try:
        with open(path, 'rb') as f:
            f.seek(0, 2)
            f.seek(max(f.tell() - 4000, 0))
            lines = f.read().decode(errors='replace').replace('\r', '\n').strip().splitlines()
            return lines[-1][-160:] if lines else ''
    except OSError:
        return ''


class Deadline:
    """Stop taking new jobs when the session is close to Kaggle's 12-hour limit."""

    def __init__(self, hours):
        self.end = time.time() + hours * 3600

    def allows(self, expected_hours):
        return time.time() + expected_hours * 3600 < self.end

In [ ]:
%%writefile r51_worker.py
"""R5.1 worker: train one inner-ensemble member per job and save what the analysis needs.

Variants (identical data, inner split, pair sampling, seed, schedule and early stopping):
  siamese    -- our objective: CE on both branches + the contrastive loss of
                snd.training.contrastive.ContrastiveLoss (main, corrected form
                same*d^2 + (1-same)*[m-d]_+^2), as in the CIFAR-10 re-run;
  ce         -- the same network and pairs with the contrastive term switched off (ratio 0),
                i.e. an ordinary cross-entropy classifier trained on the same samples;
  ce_linear  -- (optional) torchvision backbone with a plain linear head, CE only.

Per job it writes <job>.npz with, for the held-out outer fold: softmax outputs and
embeddings; for the member's own training subset: embeddings (reference set for
centroid and k-NN scores). Usage: python r51_worker.py <config.json>
"""
import copy
import json
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import snd_kaggle as K  # noqa: E402


def build_model(variant, proto):
    from snd.models.siamese import SiameseNetwork
    if variant in ('siamese', 'ce'):
        return SiameseNetwork(num_classes=10, model=proto['backbone'], embedding_dimension=proto['emb'],
                              pre_trained=proto['pre_trained'], dropout_prob=K.SIAMESE_COMMON['dropout'],
                              trainable=True, parallel=False)
    if variant == 'ce_linear':
        return LinearHead(proto)
    raise ValueError(variant)


class LinearHead(nn.Module):
    """Backbone + linear classifier; `classify` mirrors SiameseNetwork's (emb, logits) API."""

    def __init__(self, proto):
        super().__init__()
        from snd.models.siamese import BACKBONES
        build, width = BACKBONES[proto['backbone']]
        base = build(10, proto['pre_trained'])
        base.fc = nn.Flatten()
        self.feature_extractor = base
        self.fc = nn.Linear(width, 10)

    def classify(self, x):
        feat = self.feature_extractor(x)
        return feat, self.fc(feat)

    def forward(self, x1, x2):
        e1, c1 = self.classify(x1)
        e2, c2 = self.classify(x2)
        return e1, e2, c1, c2


def contrastive(emb1, emb2, same, margin):
    """snd.training.contrastive.ContrastiveLoss (Euclidean), evaluated in float32 under AMP."""
    d = nn.functional.pairwise_distance(emb1.float(), emb2.float())
    return torch.mean(same * d.pow(2) + (1 - same) * torch.clamp(margin - d, min=0).pow(2))


def run_epoch(model, loader, device, ratio, margin, ce, opt=None, scaler=None, amp=False):
    """One pass over pair batches; mirrors snd.training.trainer.Trainer.calc_loss."""
    train = opt is not None
    model.train(train)
    loss_sum, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for img1, img2, y1, y2, _, _ in loader:
            img1, img2 = img1.to(device, non_blocking=True), img2.to(device, non_blocking=True)
            y1, y2 = y1.to(device), y2.to(device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                e1, e2, c1, c2 = model(img1, img2)
                loss = ce(c1.float(), y1) + ce(c2.float(), y2)
            if ratio:
                loss = loss + ratio * contrastive(e1, e2, (y1 == y2).float(), margin)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            loss_sum += loss.item()
            correct += (c1.argmax(1) == y1).sum().item() + (c2.argmax(1) == y2).sum().item()
            total += 2 * len(y1)
    return loss_sum / len(loader), 100.0 * correct / total


@torch.no_grad()
def embed(model, ds, indices, transform, device, amp, batch=1024, workers=2):
    from snd.data.dataset import DatasetSingle
    loader = DataLoader(DatasetSingle(Subset(ds, indices), transform), batch_size=batch,
                        shuffle=False, num_workers=workers)
    model.eval()
    embs, probs = [], []
    for img, _, _ in loader:
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            e, logits = model.classify(img.to(device))
        embs.append(e.float().cpu())
        probs.append(torch.softmax(logits.float(), 1).cpu())
    return torch.cat(embs).numpy(), torch.cat(probs).numpy()


def train_member(cfg, job, ds, table, splits):
    from snd.data.dataset import DatasetPairs
    variant, outer, member = job['variant'], job['outer'], job['member']
    proto = K.PROTOCOL[(cfg['dataset'], cfg['noise'])]
    common = dict(K.SIAMESE_COMMON, **cfg.get('overrides', {}))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = cfg['amp'] and device.type == 'cuda'
    aug, plain = K.augmentation(proto['aug'])

    tr_idx, va_idx = splits[outer][member]
    seed = cfg['seed'] * 1000 + outer * 10 + member          # same pairs/init for every variant
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    workers = cfg.get('loader_workers', 2)
    train_pairs = DatasetPairs(Subset(ds, tr_idx), smart_count=False,
                               num_pairs_per_epoch=common['train_pairs'], transform=aug)
    val_pairs = DatasetPairs(Subset(ds, va_idx), smart_count=False,
                             num_pairs_per_epoch=common['val_pairs'], transform=plain)
    train_loader = DataLoader(train_pairs, batch_size=common['batch_size'], shuffle=True,
                              num_workers=workers, pin_memory=True, persistent_workers=workers > 0)
    val_loader = DataLoader(val_pairs, batch_size=512, shuffle=False, num_workers=workers,
                            pin_memory=True, persistent_workers=workers > 0)

    model = build_model(variant, proto).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=common['lr'], weight_decay=proto['wd'])
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    ce = nn.CrossEntropyLoss(label_smoothing=common['label_smoothing'])
    ratio = 1.0 if variant == 'siamese' else 0.0

    # Early stopping on validation accuracy with patience, keep the best weights
    # (Trainer.train with freeze_epoch=None).
    best_acc, best_state, stale, history = -1.0, None, 0, []
    t0 = time.time()
    for epoch in range(common['max_epochs']):
        tr_loss, tr_acc = run_epoch(model, train_loader, device, ratio, proto['margin'], ce,
                                    opt, scaler, amp)
        va_loss, va_acc = run_epoch(model, val_loader, device, ratio, proto['margin'], ce, amp=amp)
        history.append([epoch, tr_loss, tr_acc, va_loss, va_acc])
        if va_acc > best_acc:
            best_acc, stale = va_acc, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        print(f'{job["id"]} ep{epoch} train {tr_loss:.3f}/{tr_acc:.2f} val {va_loss:.3f}/{va_acc:.2f} '
              f'best {best_acc:.2f} stale {stale} [{(time.time() - t0) / 60:.0f} min]', flush=True)
        if stale >= proto['patience']:
            break
    model.load_state_dict(best_state)
    train_minutes = (time.time() - t0) / 60

    outer_idx = np.flatnonzero(table['outer_fold'].to_numpy() == outer)
    emb, probs = embed(model, ds, outer_idx, plain, device, amp, workers=workers)
    ref_emb, _ = embed(model, ds, tr_idx, plain, device, amp, workers=workers)
    out = os.path.join(cfg['out_dir'], job['id'])
    np.savez_compressed(out + '.npz', outer_idx=outer_idx, probs=probs.astype(np.float16),
                        emb=emb.astype(np.float16), ref_idx=np.asarray(tr_idx),
                        ref_emb=ref_emb.astype(np.float16), history=np.asarray(history))
    if cfg.get('save_checkpoints', True):
        torch.save({k: v.half() if v.is_floating_point() else v
                    for k, v in model.state_dict().items()}, out + '.pt')

    noisy = table['noisy_label'].to_numpy()[outer_idx]
    true = table['real_label'].to_numpy()[outer_idx]
    pred = probs.argmax(1)
    return dict(job=job['id'], variant=variant, outer=outer, member=member, epochs=len(history),
                best_val_acc=best_acc, minutes=round(train_minutes, 1),
                outer_acc_noisy=float((pred == noisy).mean()), outer_acc_true=float((pred == true).mean()))


def inner_splits(table, outer_folds, seed, n_inner):
    """Seeded StratifiedKFold over each outer-training set (original indices)."""
    from sklearn.model_selection import StratifiedKFold
    splits = {}
    folds = table['outer_fold'].to_numpy()
    noisy = table['noisy_label'].to_numpy()
    for outer in outer_folds:
        pool = np.flatnonzero(folds != outer)
        skf = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=seed + outer)
        splits[outer] = [(pool[a], pool[b]) for a, b in skf.split(pool, noisy[pool])]
    return splits


def jobs_for(cfg):
    jobs = []
    for outer in cfg['outer_folds']:
        for member in range(cfg.get('members', 10)):
            for variant in cfg['variants']:
                jobs.append(dict(id=f'{variant}_o{outer}_m{member}', variant=variant,
                                 outer=outer, member=member))
    return jobs


def main(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    K.setup_repo(cfg['code_ref'])
    proto = K.PROTOCOL[(cfg['dataset'], cfg['noise'])]
    table = K.load_label_table(K.extract_preds(K.setup_repo(cfg['code_ref']), cfg['preds_ref'],
                                               proto['preds']))
    ds = K.noisy_train_set(cfg['dataset'], table)
    table, keep = K.smoke_subsample(table, cfg)
    if keep is not None:
        ds = Subset(ds, keep)
    splits = inner_splits(table, cfg['outer_folds'], cfg['seed'], K.SIAMESE_COMMON['inner_folds'])

    queue = K.JobQueue(cfg['out_dir'])
    deadline = K.Deadline(cfg['session_hours'] - (time.time() - cfg['session_start']) / 3600)
    for job in jobs_for(cfg):
        if queue.done(job['id']):
            continue
        if not deadline.allows(cfg['member_hours']):
            print('session deadline reached; remaining jobs left for the next version', flush=True)
            break
        if not queue.claim(job['id']):
            continue
        try:
            summary = train_member(cfg, job, ds, table, splits)
        except BaseException:
            queue.release(job['id'])
            raise
        queue.finish(job['id'], summary)
        print('DONE', json.dumps(summary), flush=True)


if __name__ == '__main__':
    main(sys.argv[1])

In [ ]:
%%writefile r51_analysis.py
"""R5.1 analysis: which part of the detector carries the signal?

For every (variant, outer fold) it scores the held-out samples with
  disagreement      r(x)/m, the published detector;
  confidence        1 - mean_j p_j(y~|x), the members' mean softmax on the observed label;
  centroid_margin   mean_j [d(h_j, mu_j[y~]) - min_{k != y~} d(h_j, mu_j[k])];
  knn_disagreement  mean_j share of the k nearest training embeddings with a label != y~;
  emb_instability   std_j of d(h_j, mu_j[y~]) / s_j, s_j = member j's median own-class distance.
Embedding spaces of separately trained members are not aligned, so every embedding score is
computed inside one member (against that member's own training embeddings, labelled with
the observed labels) and only the resulting scalars are aggregated across members.
"""
import glob
import os
import re

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score

SCORES = ['disagreement', 'confidence', 'centroid_margin', 'knn_disagreement', 'emb_instability']


def load_runs(out_dir):
    runs = {}
    for path in glob.glob(os.path.join(out_dir, '*.npz')):
        variant, outer, member = re.match(r'(.+)_o(\d+)_m(\d+)\.npz', os.path.basename(path)).groups()
        runs.setdefault((variant, int(outer)), {})[int(member)] = path
    return {key: [members[m] for m in sorted(members)] for key, members in runs.items()}


def member_scores(npz, noisy_all, k=20):
    """Per-sample scalars from one member: prediction, p(y~), and the embedding scores."""
    d = np.load(npz)
    idx, ref_idx = d['outer_idx'], d['ref_idx']
    y, ref_y = noisy_all[idx], noisy_all[ref_idx]
    probs = d['probs'].astype(np.float32)
    h = torch.from_numpy(d['emb'].astype(np.float32))
    ref = torch.from_numpy(d['ref_emb'].astype(np.float32))
    classes = np.unique(ref_y)
    mu = torch.stack([ref[torch.from_numpy(ref_y == c)].mean(0) for c in classes])
    dist_mu = torch.cdist(h, mu).numpy()                         # n x c
    own = dist_mu[np.arange(len(y)), np.searchsorted(classes, y)]
    other = np.where(classes[None, :] == y[:, None], np.inf, dist_mu).min(1)
    ref_own = torch.cdist(ref, mu).numpy()[np.arange(len(ref_y)), np.searchsorted(classes, ref_y)]
    knn = np.empty(len(y))
    for start in range(0, len(y), 1024):
        nearest = torch.cdist(h[start:start + 1024], ref).topk(k, largest=False).indices.numpy()
        knn[start:start + 1024] = (ref_y[nearest] != y[start:start + 1024, None]).mean(1)
    return dict(idx=idx, pred=probs.argmax(1), p_obs=probs[np.arange(len(y)), y],
                margin=own - other, knn=knn, own_scaled=own / np.median(ref_own))


def ensemble_scores(paths, noisy_all):
    per = [member_scores(p, noisy_all) for p in paths]
    idx = per[0]['idx']
    assert all((m['idx'] == idx).all() for m in per)
    y = noisy_all[idx]
    stack = lambda key: np.stack([m[key] for m in per], 1)  # noqa: E731
    pred = stack('pred')
    return idx, pred, {
        'disagreement': (pred != y[:, None]).mean(1),
        'confidence': 1 - stack('p_obs').mean(1),
        'centroid_margin': stack('margin').mean(1),
        'knn_disagreement': stack('knn').mean(1),
        'emb_instability': stack('own_scaled').std(1) if len(per) > 1 else np.zeros(len(idx)),
    }


def best_f1(score, truth):
    order = np.argsort(-score, kind='stable')
    hits = np.cumsum(truth[order])
    k = np.arange(1, len(score) + 1)
    f1 = 2 * hits / (k + truth.sum())
    # only cut between distinct score values
    valid = np.r_[score[order][1:] != score[order][:-1], True]
    return f1[valid].max()


def detection_at(flag, truth):
    tp, fp = (flag & truth).sum(), (flag & ~truth).sum()
    fn = (~flag & truth).sum()
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / (tp + fn)
    return dict(precision=prec, recall=rec, f1=2 * tp / (2 * tp + fp + fn), fpr=fp / (~truth).sum())


def paired_bootstrap_auc(s1, s2, truth, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n):
        b = rng.integers(0, len(truth), len(truth))
        if truth[b].all() or not truth[b].any():
            continue
        diffs.append(roc_auc_score(truth[b], s1[b]) - roc_auc_score(truth[b], s2[b]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(np.mean(diffs)), float(lo), float(hi)


def analyse(out_dir, table, min_members=10):
    noisy_all = table['noisy_label'].to_numpy()
    noise_all = table['is_noisy'].to_numpy()
    rows, threshold_rows, member_rows, cache = [], [], [], {}
    for (variant, outer), paths in sorted(load_runs(out_dir).items()):
        if len(paths) < min_members:
            print(f'{variant} outer {outer}: {len(paths)} members, skipped (< {min_members})')
            continue
        idx, pred, scores = ensemble_scores(paths, noisy_all)
        truth = noise_all[idx]
        cache[(variant, outer)] = (idx, scores, truth)
        for name, s in scores.items():
            rows.append(dict(variant=variant, outer=outer, m=len(paths), score=name,
                             auc=roc_auc_score(truth, s), ap=average_precision_score(truth, s),
                             best_f1=best_f1(s, truth)))
        r = (pred != noisy_all[idx][:, None]).sum(1)
        for tau in range(1, len(paths) + 1):
            threshold_rows.append(dict(variant=variant, outer=outer, tau=tau,
                                       **detection_at(r >= tau, truth)))
        wrong = pred != noisy_all[idx][:, None]
        member_rows.append(dict(variant=variant, outer=outer,
                                miscls_rate_noisy=wrong[truth].mean(), miscls_rate_clean=wrong[~truth].mean(),
                                predicts_true_label_on_noisy=(pred[truth] == table['real_label'].to_numpy()[idx][truth, None]).mean()))

    # the published run on the same outer folds, for reference
    for outer in sorted({o for _, o in cache}):
        sub = table[table['outer_fold'] == outer]
        truth = sub['is_noisy'].to_numpy()
        s = sub['mistakes'].to_numpy() / 10
        rows.append(dict(variant='published', outer=outer, m=10, score='disagreement',
                         auc=roc_auc_score(truth, s), ap=average_precision_score(truth, s),
                         best_f1=best_f1(s, truth)))

    # paired comparisons on identical samples: variant vs variant (same score), and each
    # embedding score vs the disagreement detector (same variant)
    boot = []
    pairs = [('siamese', 'ce'), ('siamese', 'ce_linear'), ('ce', 'ce_linear')]
    for outer in sorted({o for _, o in cache}):
        for a, b in pairs:
            if (a, outer) not in cache or (b, outer) not in cache:
                continue
            i1, s1, t1 = cache[(a, outer)]
            i2, s2, _ = cache[(b, outer)]
            assert (i1 == i2).all()
            for name in SCORES:
                mean, lo, hi = paired_bootstrap_auc(s1[name], s2[name], t1)
                boot.append(dict(outer=outer, comparison=f'{a} - {b}', score=name,
                                 auc_diff=mean, ci_lo=lo, ci_hi=hi))
        for variant in ('siamese', 'ce'):
            if (variant, outer) not in cache:
                continue
            _, s, t = cache[(variant, outer)]
            for name in SCORES[1:]:
                mean, lo, hi = paired_bootstrap_auc(s[name], s['disagreement'], t)
                boot.append(dict(outer=outer, comparison=f'{variant}: {name} - disagreement',
                                 score=name, auc_diff=mean, ci_lo=lo, ci_hi=hi))
    return (pd.DataFrame(rows), pd.DataFrame(threshold_rows), pd.DataFrame(member_rows),
            pd.DataFrame(boot))


def report(out_dir, table, results_dir, min_members=10):
    os.makedirs(results_dir, exist_ok=True)
    scores, thresholds, members, boot = analyse(out_dir, table, min_members)
    for name, df in [('scores', scores), ('thresholds', thresholds), ('members', members),
                     ('bootstrap', boot)]:
        df.to_csv(os.path.join(results_dir, f'r51_{name}.csv'), index=False)
    pd.set_option('display.width', 200)
    if len(scores):
        print('\nThreshold-free detection quality (held-out outer folds):')
        print(scores.pivot_table(index=['score'], columns='variant', values=['auc', 'best_f1'])
              .round(4).to_string())
        print('\nPer-member misclassification w.r.t. the observed label (R4.3 support):')
        print(members.round(4).to_string(index=False))
    if len(boot):
        print('\nPaired bootstrap of ROC-AUC differences (95% CI):')
        print(boot.round(4).to_string(index=False))
    return scores, thresholds, members, boot

## 3. Setup

In [ ]:
import json, os, subprocess, sys, time
SESSION_START = time.time()
os.chdir('/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd())
subprocess.run('nvidia-smi -L', shell=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + PIP, check=False)
sys.path.insert(0, os.getcwd())
import snd_kaggle as K
repo = K.setup_repo(CONFIG['code_ref'])
print('code:', subprocess.run(['git', '-C', repo, 'log', '-1', '--format=%h %ad %s', '--date=short'],
                              capture_output=True, text=True).stdout)

## 4. Train (all GPUs, resumable)

In [ ]:
for train in (True, False):                     # download once, before the workers start
    K.base_dataset(CONFIG['dataset'], train)
CONFIG['out_dir'] = K.restore_previous_outputs(OUT_NAME)
CONFIG['session_start'] = SESSION_START
with open('config.json', 'w') as f:
    json.dump(CONFIG, f, indent=1)
codes = K.launch_workers(WORKER, 'config.json', poll=CONFIG.get('poll', 300))
print('worker exit codes:', codes)
done = len(K.JobQueue(CONFIG['out_dir']).summaries())
print(f'{done} of {N_JOBS} jobs finished' + ('' if done >= N_JOBS else
      ' -- commit a new version with this one attached as input to continue'))

## 5. Analysis

In [ ]:
import r51_analysis as A
table = K.load_label_table(K.extract_preds(repo, CONFIG['preds_ref'],
                                           K.PROTOCOL[(CONFIG['dataset'], CONFIG['noise'])]['preds']))
table, _ = K.smoke_subsample(table, CONFIG)
scores, thresholds, members, boot = A.report(CONFIG['out_dir'], table, 'results',
                                             min_members=CONFIG['members'])